In [1]:
import pandas as pd
import json
from pathlib import Path

In [3]:
df = pd.read_csv("CTD_pheno_term_ixns.csv")

In [4]:
# Preview structure
print("Original shape:", df.shape)
df.head()

Original shape: (423002, 13)


,chemicalname,chemicalid,casrn,phenotypename,phenotypeid,comentionedterms,organism,organismid,interaction,interactionactions,anatomyterms,inferencegenesymbols,pubmedids
0,10074-G5,C534883,NaN,ATP biosynthetic process,GO:0006754,NaN,Homo sapiens,9606.0,10074-G5 analog results in decreased ATP biosy...,decreases^phenotype,1^HL-60 Cells^D018922,NaN,26036281
1,10074-G5,C534883,NaN,ATP biosynthetic process,GO:0006754,NaN,Homo sapiens,9606.0,10074-G5 analog results in decreased ATP biosy...,decreases^phenotype,"1^Lung^D008168|2^Cell Line, Tumor^D045744",NaN,26036281
2,10074-G5,C534883,NaN,ATP biosynthetic process,GO:0006754,NaN,Homo sapiens,9606.0,10074-G5 results in decreased ATP biosynthetic...,decreases^phenotype,1^HL-60 Cells^D018922,NaN,26036281
3,10074-G5,C534883,NaN,ATP biosynthetic process,GO:0006754,NaN,Homo sapiens,9606.0,10074-G5 results in decreased ATP biosynthetic...,decreases^phenotype,"1^Lung^D008168|2^Cell Line, Tumor^D045744",NaN,26036281
4,10074-G5,C534883,NaN,cellular lipid biosynthetic process,GO:0097384,NaN,Homo sapiens,9606.0,10074-G5 analog results in increased cellular ...,increases^phenotype,"1^Lung^D008168|2^Cell Line, Tumor^D045744",NaN,26036281


In [5]:
# Drop rows missing either chemicalname or phenotypename
df = df.dropna(subset=["chemicalname", "phenotypename"])

# Drop self-loop entries
df = df[df["chemicalname"].str.strip().str.lower() != df["phenotypename"].str.strip().str.lower()]

print("After cleaning missing and self-loop rows:", df.shape)


After cleaning missing and self-loop rows: (422987, 13)


In [6]:
# Parse 'interactionactions' (e.g., 'increases^phenotype')
def extract_relation(action):
    if pd.isna(action):
        return "affects"
    return action.split("^")[0].strip().lower() or "affects"

df["Relation"] = df["interactionactions"].apply(extract_relation)

# Preview unique relation types
df["Relation"].value_counts().head()


Relation
affects      164630
decreases    163917
increases     94440
Name: count, dtype: int64

In [7]:
# Keep relevant columns
df = df[["chemicalname", "Relation", "phenotypename", "pubmedids"]]

# Drop duplicates
df = df.drop_duplicates(subset=["chemicalname", "Relation", "phenotypename"])

print("After deduplication:", df.shape)


After deduplication: (156625, 4)


In [8]:
triplets = []

for _, row in df.iterrows():
    triplets.append({
        "head": row["chemicalname"].strip(),
        "relation": row["Relation"],
        "tail": row["phenotypename"].strip(),
        "source": "CTD_pheno_term_ixns",
        "pubmed_ids": str(row["pubmedids"]).split("|") if pd.notna(row["pubmedids"]) else []
    })

print("Sample triplet:")
triplets[0]


Sample triplet:


{'head': '10074-G5',
 'relation': 'decreases',
 'tail': 'ATP biosynthetic process',
 'source': 'CTD_pheno_term_ixns',
 'pubmed_ids': ['26036281']}

In [ ]:
output_path = Path("triplets_pheno_cleaned.json")
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(triplets, f, indent=2)

print(f"Saved {len(triplets)} cleaned triplets to {output_path}")


✅ Saved 156625 cleaned triplets to triplets_pheno_cleaned.json
